In [ ]:
%matplotlib inline
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

PROJECT_ROOT = Path(r"C:\Users\asus\OneDrive\EV-projects\evcs-projects")
BENCH_DIR    = PROJECT_ROOT / "results" / "benchmarking"
EXCEL_FILE   = BENCH_DIR / "benchmark_with_SLURM.xlsx"

df = pd.read_excel(EXCEL_FILE, sheet_name="benchmark")
print(f"Loaded {len(df)} rows")
df[['Timestamp','Instance','N','T','D_km','seed','DR_best','Exact_incumbent_raw','Gap_%']]

In [ ]:
# --- build a short label for each row ---
def row_label(row, idx):
    return f"#{idx}  T={int(row['T'])} D={row['D_km']} seed={int(row['seed'])}"

df["label"] = [row_label(r, i) for i, r in df.iterrows()]

print("Labels ready.")
df[["label", "DR_best", "Exact_incumbent_raw", "Gap_%"]]


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
labels = df["label"].tolist()
x = np.arange(len(df))
width = 0.35

# â”€â”€ Panel 1: DR_best vs Exact_incumbent_raw â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
ax1 = axes[0]
bars_dr    = ax1.bar(x - width/2, df["DR_best"],              width, label="DR best",       color="tab:orange", alpha=0.85)
bars_exact = ax1.bar(x + width/2, df["Exact_incumbent_raw"],  width, label="Exact incumbent", color="tab:blue",   alpha=0.85)

# value labels on bars
for bar in bars_dr:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 2, f"{h:.1f}",
             ha="center", va="bottom", fontsize=7, color="tab:orange")
for bar in bars_exact:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 2, f"{h:.1f}",
             ha="center", va="bottom", fontsize=7, color="tab:blue")

ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
ax1.set_ylabel("Objective value (covered demand)", fontsize=9)
ax1.set_title("DR best vs Exact incumbent â€” all benchmark rows", fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(axis="y", linewidth=0.5, alpha=0.6)

# â”€â”€ Panel 2: Gap_% bar chart â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
ax2 = axes[1]
gaps   = df["Gap_%"].tolist()
colors = ["tab:green" if g <= 1.0 else "tab:orange" if g <= 5.0 else "tab:red"
          for g in gaps]
bars_gap = ax2.bar(x, gaps, color=colors, alpha=0.85, edgecolor="white")

# value labels
for bar, g in zip(bars_gap, gaps):
    va  = "bottom" if g >= 0 else "top"
    off = 0.05 if g >= 0 else -0.05
    ax2.text(bar.get_x() + bar.get_width()/2, g + off, f"{g:.2f}%",
             ha="center", va=va, fontsize=8)

ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
ax2.set_ylabel("Gap  (Exact âˆ’ DR) / Exact  [%]", fontsize=9)
ax2.set_title("Optimality gap per run   (green â‰¤ 1%  |  orange â‰¤ 5%  |  red > 5%)", fontsize=11)
ax2.grid(axis="y", linewidth=0.5, alpha=0.6)

# legend patches
from matplotlib.patches import Patch
ax2.legend(handles=[
    Patch(color="tab:green",  label="gap â‰¤ 1%"),
    Patch(color="tab:orange", label="1% < gap â‰¤ 5%"),
    Patch(color="tab:red",    label="gap > 5%"),
], fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# --- summary table with color-coded gap ---
cols = ['Timestamp','Instance','Policy','N','T','D_km','seed',
        'Exact_incumbent_raw','DR_best','Gap_%','DR_iters','DR_time_s','Exact_time_s']
show = [c for c in cols if c in df.columns]
df[show].style \
    .format({'Exact_incumbent_raw':'{:.4f}','DR_best':'{:.4f}',
             'Gap_%':'{:.4f}%','DR_time_s':'{:.1f}s','Exact_time_s':'{:.1f}s'}) \
    .background_gradient(subset=['Gap_%'], cmap='RdYlGn_r')
